# DPPO robustness study on Robomimic Square

**Before you start:** open the Resources panel and check your compute units. The simulator is CPU-heavy, so an **L4** runtime is usually better value than an A100, which burns units much faster for little speed-up here (cell 2 prints your vCPU count). My rough estimate is 10-16 runtime hours for the full list. The status table shows the real projection once training starts.

**How to run:** pick a GPU runtime (Runtime > Change runtime type: **L4** or **A100**, High-RAM on), then **Runtime > Run all**.
You'll get one Google Drive permission pop-up. After that, everything runs on its own and saves to
`MyDrive/DPPO_Robustness/`.

**What it runs (in this order):**
1. Setup and checks (about 10-15 min): installs DPPO, downloads the pretrained Square policy, then tests the perturbation code and one tiny training run.
2. Pretrained policy evaluation, nominal plus one perturbation, as a quick sanity check.
3. **DPPO fine-tuning**, seed 42. 20 envs, 1000 steps per iteration, 41 iterations.
4. Robustness sweep: pretrained and DPPO under 11 conditions (nominal; nut shift 2/4/6 cm; observation noise 0.02/0.05/0.1; friction x0.5/x2; nut mass x0.5/x2), 100 episodes each.
5. **Robust DPPO** (domain randomization during fine-tuning), then the same sweep.
6. Demo videos, then extra seed 43 for both methods if time allows.

**If the runtime disconnects:** reconnect and **Run all** again. Finished jobs are skipped, and training resumes from its last checkpoint (saved every 5 iterations).

**Live status:** the last cell of Part 1 prints a status table every 5 minutes. The same table is in `MyDrive/DPPO_Robustness/pipeline/STATUS_ALL.md`.

**Control:** create an empty file `pipeline/STOP` on Drive to stop after the current job. Create `pipeline/SKIP_<job name>` to skip a job.

Part 2 (analysis) can be run at any time, even while Part 1 is still going. It only reports what has actually finished.

## Part 1 - Setup and overnight runs
### 1. Settings (the only cell you normally edit)

In [ ]:
# "ALL" = this one runtime does everything.
# Two runtimes at once (only if you have the compute units): set "A" in one notebook copy and "B" in the other.
#   A = pretrained + DPPO side and the videos, B = Robust DPPO side. Both share the same Drive folder.
ROLE = "ALL"

PROJECT = "/content/drive/MyDrive/DPPO_Robustness"

# Fine-tuning size: 20 envs x 1000 steps x 41 iterations (2.88M env steps per run)
TRAIN = dict(n_envs=20, n_steps=1000, n_itr=41, save_every=5, val_freq=10)
SEEDS_MAIN = [42]    # training seed for the main comparison
SEEDS_EXTRA = [43]   # stretch goal, runs last

# Evaluation: 25 parallel envs x 4 episodes = 100 episodes per condition, same initial states for every method
EVAL = dict(n_envs=25, episodes_per_env=4, seed=1000, video_envs=4)

# Robust DPPO: what gets randomized every training episode
DR = dict(dr_shift_max_cm=4.0, dr_noise_max=0.05, dr_scale_low=0.5, dr_scale_high=2.0)

# Test conditions (nut shift 6 cm and noise 0.1 are outside the Robust DPPO training ranges on purpose)
CONDITIONS = [dict(name="nominal", label="Nominal", type="Nominal", level=0, unit="", overrides={},
                   level_for={"Object shift": 0, "Observation noise": 0, "Friction": 1.0, "Nut mass": 1.0})]
for v in (2, 4, 6):
    CONDITIONS.append(dict(name=f"shift_{v}cm", label=f"Nut shifted {v} cm", type="Object shift", level=v,
                           unit="nut shift (cm)", overrides={"shift_cm": float(v)}))
for v in (0.02, 0.05, 0.1):
    CONDITIONS.append(dict(name=f"noise_{v}", label=f"Obs. noise std {v}", type="Observation noise", level=v,
                           unit="noise std (normalized state)", overrides={"obs_noise": v}))
for v in (0.5, 2.0):
    CONDITIONS.append(dict(name=f"friction_x{v}", label=f"Friction x{v}", type="Friction", level=v,
                           unit="friction scale (all surfaces)", overrides={"friction_scale": v}))
for v in (0.5, 2.0):
    CONDITIONS.append(dict(name=f"mass_x{v}", label=f"Nut mass x{v}", type="Nut mass", level=v,
                           unit="nut mass scale", overrides={"mass_scale": v}))
print(len(CONDITIONS), "conditions:", [c["name"] for c in CONDITIONS])

### 2. Drive and hardware check

In [ ]:
import os, json, subprocess, sys, time, glob, shutil
from google.colab import drive

drive.mount("/content/drive")
for d in ["pipeline", "data/robomimic/square", "runs"]:
    os.makedirs(f"{PROJECT}/{d}", exist_ok=True)
os.makedirs("/content/work", exist_ok=True)


def sh(cmd, check=True, quiet=False):
    print("$", cmd[:300])
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if not quiet or r.returncode:
        print((r.stdout or "")[-3000:], (r.stderr or "")[-3000:])
    if check and r.returncode:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd[:200]}")
    return r


gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, text=True,
                     capture_output=True).stdout.strip() or "NO GPU"
cpus = os.cpu_count()
ram_gb = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9)
HARDWARE = dict(gpu=gpu, cpus=cpus, ram_gb=ram_gb)
print(HARDWARE)
if gpu == "NO GPU":
    raise SystemExit("No GPU: Runtime > Change runtime type > L4 or A100, then Run all again.")
if cpus < 8:
    print("WARNING: fewer than 8 vCPUs. The simulator is CPU-bound, so training will be slow. "
          "Turn on High-RAM or pick A100/L4 if you can.")
if ram_gb < 40:
    print("WARNING: under 40 GB RAM. If evaluations crash with 'Killed', lower EVAL['n_envs'] to 10 "
          "and raise EVAL['episodes_per_env'] to 10.")

### 3. Install (about 5-10 min on a fresh runtime, skipped if already done)
Separate Python 3.10 environment with library versions I know work together: torch 2.4.0, robosuite 1.4.1, robomimic 0.5.0, mujoco 3.1.6, numpy 1.26.4.

In [ ]:
VENV, REPO = "/content/venv", "/content/dppo"
VPY = f"{VENV}/bin/python"
OK = f"{VENV}/.install_ok"

LANG_STUB = '''# robomimic 0.5 loads a large CLIP language model at import time in every simulator
# process. We never use language inputs, so this stub keeps the same interface without the model.
LANG_EMB_OBS_KEY = "lang_emb"

def get_lang_emb(lang):
    if lang is None:
        return None
    raise RuntimeError("language embeddings are disabled in this setup")

def get_lang_emb_shape():
    return [768]
'''

if not os.path.exists(OK):
    sh("apt-get -qq update && apt-get -qq install -y libegl1 libgl1 libglib2.0-0 libosmesa6 > /dev/null", check=False)
    sh("pip -q install uv")
    if not os.path.exists(REPO):
        sh(f"git clone -q https://github.com/irom-princeton/dppo.git {REPO}")
    sh(f"cd {REPO} && git checkout -q cc7234ad7ff39a8f32de3af903606723a16f0648")  # version we checked
    sh(f"uv venv {VENV} --python 3.10 --quiet")
    sh(f"cd {REPO} && uv pip install --quiet --python {VPY} -e .")
    sh(f'uv pip install --quiet --python {VPY} "numpy==1.26.4" "mujoco==3.1.6" numba scipy pillow '
       f'opencv-python-headless termcolor h5py psutil tqdm imageio-ffmpeg')
    sh(f'uv pip install --quiet --python {VPY} --no-deps '
       f'"robosuite @ git+https://github.com/ARISE-Initiative/robosuite.git@v1.4.1" '
       f'"robomimic @ git+https://github.com/ARISE-Initiative/robomimic.git@v0.5.0"')
    site = subprocess.check_output([VPY, "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"], text=True).strip()
    with open(f"{site}/robomimic/utils/lang_utils.py", "w") as f:
        f.write(LANG_STUB)
    with open(f"{site}/egl_probe.py", "w") as f:  # robomimic asks egl_probe for a GPU only when rendering videos
        f.write("def get_available_devices():\n    return [0]\n")
    open(OK, "w").write("ok")

# pick the OpenGL backend that works (only matters for videos)
MUJOCO_GL = None
for gl in ("egl", "osmesa"):
    r = sh(f'cd {REPO} && MUJOCO_GL={gl} PYOPENGL_PLATFORM={gl} {VPY} -c "import torch, robosuite, mujoco, gym, hydra, '
           f'robomimic.utils.env_utils; print(torch.__version__, torch.cuda.is_available(), robosuite.__version__, mujoco.__version__)"',
           check=False)
    if r.returncode == 0 and "True" in r.stdout:
        MUJOCO_GL = gl
        break
assert MUJOCO_GL, "Imports failed - see the error above."
print("Install OK, MUJOCO_GL =", MUJOCO_GL)

### 4. Our additions to DPPO
1. Perturbation wrapper: nut shift, observation noise, friction and mass, plus domain randomization for Robust DPPO.
2. Resumable training launcher.
3. A self-test showing the perturbations do what they are supposed to do.

Upstream DPPO code is not edited, apart from one registration line.

In [ ]:
%%writefile /content/work/perturb.py
"""
DPPO's Robomimic low-dim wrapper with optional perturbations, used for our robustness tests.

Everything is off by default, and then it behaves exactly like the original wrapper
(no extra random draws, no physics changes).

Fixed perturbations for testing (same level every episode):
  shift_cm        move the nut this many cm in a random direction after reset
  obs_noise       std of Gaussian noise added to the normalized 23-dim state
  friction_scale  multiply the sliding friction of every geom in the scene
  mass_scale      multiply the mass of the nut

Domain randomization for Robust DPPO (dr=True), new values every episode:
  shift ~ U(0, dr_shift_max_cm), noise std ~ U(0, dr_noise_max),
  friction and mass ~ log-uniform(dr_scale_low, dr_scale_high)
"""

import numpy as np

from .robomimic_lowdim import RobomimicLowdimWrapper


class PerturbedRobomimicLowdimWrapper(RobomimicLowdimWrapper):
    def __init__(
        self,
        env,
        shift_cm=0.0,
        obs_noise=0.0,
        friction_scale=1.0,
        mass_scale=1.0,
        dr=False,
        dr_shift_max_cm=4.0,
        dr_noise_max=0.05,
        dr_scale_low=0.5,
        dr_scale_high=2.0,
        **kwargs,
    ):
        super().__init__(env, **kwargs)
        self.shift_cm = float(shift_cm)
        self.obs_noise = float(obs_noise)
        self.friction_scale = float(friction_scale)
        self.mass_scale = float(mass_scale)
        self.dr = bool(dr)
        self.dr_shift_max_cm = float(dr_shift_max_cm)
        self.dr_noise_max = float(dr_noise_max)
        self.dr_scale_low = float(dr_scale_low)
        self.dr_scale_high = float(dr_scale_high)
        self.active = (
            self.dr
            or self.shift_cm > 0
            or self.obs_noise > 0
            or self.friction_scale != 1.0
            or self.mass_scale != 1.0
        )
        self._rng = None
        self._orig = None
        self._noise_now = 0.0
        self.last_perturbation = {}

    def _robosuite_env(self):
        return self.env.env  # robomimic EnvRobosuite -> robosuite NutAssemblySquare

    def _setup_once(self):
        rs = self._robosuite_env()
        sim = rs.sim
        if self._orig is None:
            nut = rs.nuts[rs.nut_id]
            self._nut_joint = nut.joints[0]
            self._nut_body = rs.obj_body_id[nut.name]
            self._orig = {
                "friction": np.array(sim.model.geom_friction[:, 0]).copy(),
                "mass": float(sim.model.body_mass[self._nut_body]),
            }
        if self._rng is None:
            # seeded from numpy's global state, which DPPO seeds per env (seed + env index)
            self._rng = np.random.RandomState(np.random.randint(2**31 - 1))

    def _draw(self):
        if self.dr:
            shift = self._rng.uniform(0.0, self.dr_shift_max_cm)
            noise = self._rng.uniform(0.0, self.dr_noise_max)
            lo, hi = np.log(self.dr_scale_low), np.log(self.dr_scale_high)
            fric = float(np.exp(self._rng.uniform(lo, hi)))
            mass = float(np.exp(self._rng.uniform(lo, hi)))
        else:
            shift, noise = self.shift_cm, self.obs_noise
            fric, mass = self.friction_scale, self.mass_scale
        angle = self._rng.uniform(0.0, 2.0 * np.pi)
        return shift, angle, noise, fric, mass

    def _apply_physics(self, shift_cm, angle, fric, mass):
        sim = self._robosuite_env().sim
        sim.model.geom_friction[:, 0] = self._orig["friction"] * fric
        sim.model.body_mass[self._nut_body] = self._orig["mass"] * mass
        if shift_cm > 0:
            qpos = np.array(sim.data.get_joint_qpos(self._nut_joint), dtype=float).copy()
            qpos[0] += shift_cm / 100.0 * np.cos(angle)
            qpos[1] += shift_cm / 100.0 * np.sin(angle)
            sim.data.set_joint_qpos(self._nut_joint, qpos)
        sim.forward()

    def _noisy(self, obs):
        if self._noise_now > 0:
            state = obs["state"]
            obs["state"] = state + self._rng.normal(0.0, self._noise_now, size=state.shape).astype(state.dtype)
        return obs

    def reset(self, options={}, **kwargs):
        obs = super().reset(options=options, **kwargs)
        if not self.active:
            return obs
        self._setup_once()
        shift, angle, noise, fric, mass = self._draw()
        self._apply_physics(shift, angle, fric, mass)
        self._noise_now = noise
        self.last_perturbation = dict(shift_cm=shift, angle=angle, obs_noise=noise, friction_scale=fric, mass_scale=mass)
        obs = self.get_observation(self.env.get_observation())  # re-read the state after moving the nut
        return self._noisy(obs)

    def step(self, action):
        obs, reward, done, info = super().step(action)
        if self.active:
            obs = self._noisy(obs)
        return obs, reward, done, info

In [ ]:
%%writefile /content/work/dppo_run_resumable.py
"""
Same as `python script/run.py ...`, but a fine-tuning run picks up from the newest
checkpoint in its logdir if one exists (so a Colab disconnect only loses the iterations
since the last save). Run from the DPPO repo root.

Note: optimizer state and the running reward scaler are not stored in DPPO checkpoints,
so they restart after a resume; the policy and critic weights continue exactly.
"""

import glob
import os
import re
import runpy
import sys

REPO = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.join(REPO, "script"))  # for `from download_url import ...` in run.py
sys.path.insert(0, REPO)

import agent.finetune.train_agent as train_agent  # noqa: E402

_original_init = train_agent.TrainAgent.__init__


def _init_with_resume(self, cfg):
    _original_init(self, cfg)
    found = []
    for path in glob.glob(os.path.join(self.checkpoint_dir, "state_*.pt")):
        m = re.search(r"state_(\d+)\.pt$", path)
        if m:
            found.append(int(m.group(1)))
    for itr in sorted(found, reverse=True):
        try:
            self.load(itr)
        except Exception as e:  # half-written file from a crash
            bad = os.path.join(self.checkpoint_dir, f"state_{itr}.pt")
            os.replace(bad, bad + ".corrupt")
            print(f"[resume] state_{itr}.pt could not be loaded ({e}); trying an older one", flush=True)
            continue
        self.itr = itr + 1
        print(f"[resume] loaded state_{itr}.pt, continuing at iteration {self.itr}", flush=True)
        return
    print("[resume] no checkpoint found, starting from iteration 0", flush=True)


train_agent.TrainAgent.__init__ = _init_with_resume

sys.argv[0] = os.path.join(REPO, "script", "run.py")
runpy.run_path(sys.argv[0], run_name="__main__")

In [ ]:
%%writefile /content/work/perturb_selftest.py
"""
Checks that the perturbations do what they are supposed to do. Run from the DPPO repo root:
    python perturb_selftest.py <normalization.npz> <out.json>
"""

import json
import sys

import numpy as np
import robomimic.utils.env_utils as EnvUtils
import robomimic.utils.obs_utils as ObsUtils

from env.gym_utils.wrapper.robomimic_lowdim import RobomimicLowdimWrapper
from env.gym_utils.wrapper.perturb import PerturbedRobomimicLowdimWrapper
from env.gym_utils.wrapper import wrapper_dict

NORM, OUT = sys.argv[1], sys.argv[2]
KEYS = ["robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos", "object"]
ObsUtils.initialize_obs_modality_mapping_from_dict({"low_dim": KEYS})
with open("cfg/robomimic/env_meta/square.json") as f:
    META = json.load(f)
META["reward_shaping"] = False


def make(cls=PerturbedRobomimicLowdimWrapper, **kw):
    env = EnvUtils.create_env_from_metadata(env_meta=META, render=False, render_offscreen=False, use_image_obs=False)
    env.env.hard_reset = False
    return cls(env, normalization_path=NORM, low_dim_keys=KEYS, **kw)


def nut_xy(w):
    rs = w.env.env
    return rs.sim.data.body_xpos[rs.obj_body_id[rs.nuts[rs.nut_id].name]][:2].copy()


checks = {}
checks["registered"] = wrapper_dict["robomimic_lowdim"] is PerturbedRobomimicLowdimWrapper

orig, ours = make(RobomimicLowdimWrapper), make()
orig.seed(7)
o1 = orig.reset()
ours.seed(7)
o2 = ours.reset()
checks["defaults_identical_obs"] = bool(np.allclose(o1["state"], o2["state"]))
checks["defaults_identical_nut_xy"] = bool(np.allclose(nut_xy(orig), nut_xy(ours)))

shifted = make(shift_cm=4.0)
shifted.seed(7); shifted.reset()
d = float(np.linalg.norm(nut_xy(shifted) - nut_xy(orig)) * 100)
checks["shift_4cm_measured_cm"] = round(d, 4)
checks["shift_ok"] = abs(d - 4.0) < 0.05

phys = make(friction_scale=2.0, mass_scale=0.5)
phys.seed(7); phys.reset()
sim, base_sim = phys.env.env.sim, orig.env.env.sim
fr = np.array(sim.model.geom_friction[:, 0]) / np.maximum(np.array(base_sim.model.geom_friction[:, 0]), 1e-9)
b = phys._nut_body
checks["friction_ratio_mean"] = round(float(fr.mean()), 4)
checks["mass_ratio"] = round(float(sim.model.body_mass[b] / base_sim.model.body_mass[b]), 4)
checks["physics_ok"] = abs(checks["friction_ratio_mean"] - 2.0) < 1e-3 and abs(checks["mass_ratio"] - 0.5) < 1e-3

noisy = make(obs_noise=0.05)
noisy.seed(7)
o3 = noisy.reset()
checks["noise_std_measured"] = round(float(np.std(o3["state"] - o1["state"])), 4)

dr = make(dr=True)
dr.seed(3)
draws = []
for _ in range(20):
    dr.reset()
    draws.append(dr.last_perturbation)
checks["dr_shift_range_cm"] = [round(min(x["shift_cm"] for x in draws), 3), round(max(x["shift_cm"] for x in draws), 3)]
checks["dr_friction_range"] = [round(min(x["friction_scale"] for x in draws), 3), round(max(x["friction_scale"] for x in draws), 3)]
checks["dr_ok"] = all(0 <= x["shift_cm"] <= 4.0 and 0.5 <= x["friction_scale"] <= 2.0 and 0.5 <= x["mass_scale"] <= 2.0 for x in draws)

for w in (orig, shifted, phys, noisy, dr):
    for _ in range(20):
        w.step(np.zeros(7))
checks["steps_ok"] = True

checks["ALL_OK"] = bool(
    checks["registered"] and checks["defaults_identical_obs"] and checks["defaults_identical_nut_xy"]
    and checks["shift_ok"] and checks["physics_ok"] and checks["dr_ok"]
)
with open(OUT, "w") as f:
    json.dump(checks, f, indent=2)
print(json.dumps(checks, indent=2))

In [ ]:
shutil.copy("/content/work/perturb.py", f"{REPO}/env/gym_utils/wrapper/perturb.py")
shutil.copy("/content/work/dppo_run_resumable.py", f"{REPO}/dppo_run_resumable.py")
shutil.copy("/content/work/perturb_selftest.py", f"{REPO}/perturb_selftest.py")
init = f"{REPO}/env/gym_utils/wrapper/__init__.py"
reg = ("\n# same wrapper plus optional perturbations (all off by default)\n"
       "from .perturb import PerturbedRobomimicLowdimWrapper\n"
       "wrapper_dict[\"robomimic_lowdim\"] = PerturbedRobomimicLowdimWrapper\n")
if "PerturbedRobomimicLowdimWrapper" not in open(init).read():
    open(init, "a").write(reg)
sh(f"cd {REPO} && git status --short && git diff --stat")
# keep a copy of our code next to the results
os.makedirs(f"{PROJECT}/code", exist_ok=True)
for f in ["perturb.py", "dppo_run_resumable.py", "perturb_selftest.py"]:
    shutil.copy(f"/content/work/{f}", f"{PROJECT}/code/{f}")

### 5. Pretrained Square policy and normalization stats (downloaded once to Drive)
These are the DPPO authors' files, We also save its EMA weights as a separate file, because DPPO fine-tunes from the EMA weights, while its evaluation script reads the raw weights.

In [ ]:
PRE_DIR = f"{PROJECT}/log/robomimic-pretrain/square/square_pre_diffusion_mlp_ta4_td20/2024-07-10_01-46-16/checkpoint"
PRE = f"{PRE_DIR}/state_8000.pt"
PRE_EMA = f"{PRE_DIR}/state_8000_ema_as_model.pt"
NORM = f"{PROJECT}/data/robomimic/square/normalization.npz"
os.makedirs(PRE_DIR, exist_ok=True)
links = {PRE: "https://drive.google.com/file/d/1lP9mNe2AxMigfOywcaHOOR7FxQ-KR_Ee/view",
         NORM: "https://drive.google.com/file/d/1FFMqWVv0145OJjbA_iglkWywmdK22Za-/view"}
for path, url in links.items():
    if not os.path.exists(path) or os.path.getsize(path) < 1000:
        r = sh(f"{VENV}/bin/gdown --fuzzy '{url}' -O '{path}'", check=False)
        if r.returncode or not os.path.exists(path):
            print(f"\nDOWNLOAD FAILED (Google Drive quota is common). Fix by hand: open {url}, download the file, and upload it to\n  {path}\nThen run this cell again.")
            raise SystemExit
sh(f'''{VPY} -c "
import torch
c = torch.load('{PRE}', map_location='cpu', weights_only=True)
print('checkpoint keys:', list(c.keys()))
torch.save({{'model': c['ema']}}, '{PRE_EMA}')
print('saved EMA copy with', sum(v.numel() for k, v in c['ema'].items() if k.startswith('network.')), 'network parameters')
"''')

### 6. Smoke tests (about 5-8 min). Don't start the overnight run unless all four pass.

In [ ]:
ENV_VARS = dict(DPPO_DATA_DIR=f"{PROJECT}/data", DPPO_LOG_DIR=f"{PROJECT}/log", DPPO_WANDB_ENTITY="none",
                WANDB_MODE="disabled", MUJOCO_GL=MUJOCO_GL, PYOPENGL_PLATFORM=MUJOCO_GL, OMP_NUM_THREADS="1",
                MKL_NUM_THREADS="1", HYDRA_FULL_ERROR="1", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
envs = " ".join(f"{k}={v}" for k, v in ENV_VARS.items())
SMOKE = "/content/smoke"
shutil.rmtree(SMOKE, ignore_errors=True)

# (a) perturbations do what we claim
sh(f"cd {REPO} && {envs} {VPY} perturb_selftest.py {NORM} {PROJECT}/pipeline/perturb_selftest.json", quiet=True)
st = json.load(open(f"{PROJECT}/pipeline/perturb_selftest.json"))
print(json.dumps(st, indent=1))
assert st["ALL_OK"], "perturbation self-test failed"

# (b) one short evaluation through DPPO's own script, with a perturbation switched on
t = time.time()
sh(f"cd {REPO} && {envs} {VPY} script/run.py --config-dir=cfg/robomimic/eval/square --config-name=eval_diffusion_mlp "
   f"base_policy_path={PRE_EMA} ft_denoising_steps=0 model._target_=model.diffusion.diffusion_eval.DiffusionEval "
   f"env.n_envs=4 n_steps=100 seed=1 logdir={SMOKE}/eval +env.wrappers.robomimic_lowdim.shift_cm=4.0 > {SMOKE}_eval.log 2>&1 "
   f"|| (tail -40 {SMOKE}_eval.log; false)")
print("eval smoke OK in", round(time.time() - t), "s:", sh(f"grep 'eval: num episode' {SMOKE}_eval.log", quiet=True).stdout)

# (c) two tiny fine-tuning iterations, then a third after a restart, to check training and resume
for n_itr in (2, 3):
    sh(f"cd {REPO} && {envs} {VPY} dppo_run_resumable.py --config-dir=cfg/robomimic/finetune/square "
       f"--config-name=ft_ppo_diffusion_mlp logdir={SMOKE}/train seed=1 wandb=null base_policy_path={PRE} "
       f"env.n_envs=4 train.n_steps=50 train.n_train_itr={n_itr} train.save_model_freq=1 train.val_freq=10 "
       f"+env.wrappers.robomimic_lowdim.dr=true > {SMOKE}_train{n_itr}.log 2>&1 || (tail -40 {SMOKE}_train{n_itr}.log; false)")
print(sh(f"grep -h 'resume\\]\\|: step' {SMOKE}_train2.log {SMOKE}_train3.log", quiet=True).stdout)
assert "loaded state_1.pt" in open(f"{SMOKE}_train3.log").read(), "resume did not work"

# (d) evaluating a fine-tuned checkpoint (base + fine-tuned denoising steps)
sh(f"cd {REPO} && {envs} {VPY} script/run.py --config-dir=cfg/robomimic/eval/square --config-name=eval_diffusion_mlp "
   f"base_policy_path={SMOKE}/train/checkpoint/state_2.pt ft_denoising_steps=10 "
   f"model._target_=model.diffusion.diffusion_eval.DiffusionEval env.n_envs=4 n_steps=100 seed=1 logdir={SMOKE}/eval_ft "
   f"> {SMOKE}_evalft.log 2>&1 || (tail -40 {SMOKE}_evalft.log; false)")
assert os.path.exists(f"{SMOKE}/eval_ft/result.npz")
print("ALL SMOKE TESTS PASSED - continue")

### 7. Start the overnight pipeline (runs in the background; safe to run again)

In [ ]:
%%writefile /content/work/pipeline.py
"""
Runs all the training and evaluation jobs for the DPPO robustness study, one after another.

    python pipeline.py config.json

Runs the job list in priority order, one job at a time. Finished jobs are detected from
files on Google Drive, so after a disconnect you just start it again and it carries on.
Every minute it writes STATUS_<role>.md and status_<role>.json into <project>/pipeline/.

Control files (create them in <project>/pipeline/ from Drive or the notebook):
    STOP              finish the current job, then exit
    SKIP_<job name>   never run that job
"""

import datetime as dt
import glob
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import time

import numpy as np

CFG = json.load(open(sys.argv[1]))
ROLE = CFG["role"]
P = CFG["project"]
REPO = CFG["repo"]
PY = CFG["python"]
T = CFG["train"]
E = CFG["eval"]
PIPE_DIR = os.path.join(P, "pipeline")
os.makedirs(PIPE_DIR, exist_ok=True)
STATUS_JSON = os.path.join(PIPE_DIR, f"status_{ROLE}.json")
STATUS_MD = os.path.join(PIPE_DIR, f"STATUS_{ROLE}.md")
EVENTS = os.path.join(PIPE_DIR, f"events_{ROLE}.log")
EP_LEN = CFG["max_episode_steps"] // CFG["act_steps"]  # policy calls per episode (400 / 4 = 100)

ENV = dict(os.environ)
ENV.update(CFG["env_vars"])

try:
    from zoneinfo import ZoneInfo
    TZ = ZoneInfo(CFG.get("timezone", "America/Phoenix"))
except Exception:
    TZ = None


def now():
    return dt.datetime.now(TZ) if TZ else dt.datetime.now()


def stamp(t=None):
    return (t or now()).strftime("%Y-%m-%d %H:%M:%S")


def event(msg):
    line = f"[{stamp()}] {msg}"
    print(line, flush=True)
    with open(EVENTS, "a") as f:
        f.write(line + "\n")


# Job list
def train_dir(kind, seed):
    return os.path.join(P, "runs", "train", f"{kind}_s{seed}")


def final_ckpt(kind, seed):
    return os.path.join(train_dir(kind, seed), "checkpoint", f"state_{T['n_itr'] - 1}.pt")


def train_job(kind, seed):
    logdir = train_dir(kind, seed)
    cmd = [
        PY, "dppo_run_resumable.py",
        "--config-dir=cfg/robomimic/finetune/square", "--config-name=ft_ppo_diffusion_mlp",
        f"logdir={logdir}", f"seed={seed}", "wandb=null",
        f"base_policy_path={CFG['pretrained_ckpt']}",
        f"env.n_envs={T['n_envs']}", f"train.n_steps={T['n_steps']}", f"train.n_train_itr={T['n_itr']}",
        f"train.save_model_freq={T['save_every']}", f"train.val_freq={T['val_freq']}",
    ]
    if kind == "robust":
        cmd.append("+env.wrappers.robomimic_lowdim.dr=true")
        cmd += [f"+env.wrappers.robomimic_lowdim.{k}={v}" for k, v in CFG["dr"].items()]
    return dict(name=f"train_{kind}_s{seed}", kind="train", cmd=cmd, logdir=logdir,
                done_file=os.path.join(logdir, "DONE.json"), deps=[], model=f"{kind}_s{seed}",
                final_ckpt=final_ckpt(kind, seed))


def model_ckpt(model):
    if model == "pretrained":
        return CFG["pretrained_ema_ckpt"], 0, []
    if model == "pretrained_raw":
        return CFG["pretrained_ckpt"], 0, []
    kind, seed = model.rsplit("_s", 1)
    return final_ckpt(kind, int(seed)), CFG["ft_denoising_steps"], [f"train_{model}"]


def eval_job(model, cond, video=False):
    ckpt, ftd, deps = model_ckpt(model)
    sub = "video" if video else "eval"
    logdir = os.path.join(P, "runs", sub, model, cond["name"])
    n_envs = E["video_envs"] if video else E["n_envs"]
    n_steps = EP_LEN * (1 if video else E["episodes_per_env"])
    cmd = [
        PY, "script/run.py",
        "--config-dir=cfg/robomimic/eval/square", "--config-name=eval_diffusion_mlp",
        f"base_policy_path={ckpt}", f"ft_denoising_steps={ftd}",
        "model._target_=model.diffusion.diffusion_eval.DiffusionEval",
        f"env.n_envs={n_envs}", f"n_steps={n_steps}", f"seed={E['seed']}", f"logdir={logdir}",
    ]
    cmd += [f"+env.wrappers.robomimic_lowdim.{k}={v}" for k, v in cond["overrides"].items()]
    if video:
        cmd += [f"render_num={n_envs}", "env.save_video=true"]
    return dict(name=f"{sub}_{model}__{cond['name']}", kind=sub, cmd=cmd, logdir=logdir,
                done_file=os.path.join(logdir, "result.npz"), deps=deps, model=model, cond=cond["name"])


def build_jobs():
    """Priority order. Role A = pretrained + DPPO side, role B = Robust DPPO side, ALL = both."""
    conds = CFG["conditions"]
    by_name = {c["name"]: c for c in conds}
    nominal, probe = by_name["nominal"], by_name.get("shift_4cm", conds[1])
    s0 = CFG["seeds_main"][0]
    # 1) sanity: pretrained policy (nominal + one perturbation) and the raw-weights check
    # 2) DPPO fine-tuning, then the full robustness sweep of pretrained and DPPO
    main_a = [eval_job("pretrained", nominal), eval_job("pretrained_raw", nominal), eval_job("pretrained", probe),
              train_job("dppo", s0)]
    main_a += [eval_job("pretrained", c) for c in conds if c["name"] not in ("nominal", probe["name"])]
    main_a += [eval_job(f"dppo_s{s0}", c) for c in conds]
    # 3) Robust DPPO (domain randomization during fine-tuning) and its sweep
    main_b = [train_job("robust", s0)] + [eval_job(f"robust_s{s0}", c) for c in conds]
    # 4) demo videos
    videos = [eval_job("pretrained", nominal, True), eval_job(f"dppo_s{s0}", nominal, True),
              eval_job(f"dppo_s{s0}", probe, True), eval_job(f"robust_s{s0}", probe, True)]
    # 5) extra training seeds (stretch goal)
    extra_a, extra_b = [], []
    for s in CFG["seeds_extra"]:
        extra_a += [train_job("dppo", s)] + [eval_job(f"dppo_s{s}", c) for c in conds]
        extra_b += [train_job("robust", s)] + [eval_job(f"robust_s{s}", c) for c in conds]
    if ROLE == "A":
        return main_a + videos + extra_a
    if ROLE == "B":
        return main_b + extra_b
    return main_a + main_b + videos + extra_a + extra_b


JOBS = build_jobs()
ALL_NAMES = {j["name"] for j in JOBS}


# State helpers
def is_done(job_or_name):
    if isinstance(job_or_name, str):  # a dependency, possibly owned by the other runtime
        if job_or_name.startswith("train_"):
            kind, seed = job_or_name[len("train_"):].rsplit("_s", 1)
            return os.path.exists(os.path.join(train_dir(kind, int(seed)), "DONE.json"))
        return False
    job = job_or_name
    if job["kind"] == "video":
        return os.path.exists(job["done_file"]) and bool(glob.glob(os.path.join(job["logdir"], "render", "*.mp4")))
    return os.path.exists(job["done_file"])


def skipped(job):
    return os.path.exists(os.path.join(PIPE_DIR, f"SKIP_{job['name']}"))


def tail(path, n_bytes=200_000):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            size = f.tell()
            f.seek(max(0, size - n_bytes))
            return f.read().decode("utf-8", "replace")
    except FileNotFoundError:
        return ""


def last_ckpt_itr(logdir):
    its = []
    for p in glob.glob(os.path.join(logdir, "checkpoint", "state_*.pt")):
        m = re.search(r"state_(\d+)\.pt$", p)
        if m:
            its.append(int(m.group(1)))
    return max(its) if its else None


def prune_checkpoints(logdir, keep=2):
    """Keep the newest `keep` checkpoints plus the final one, to save Drive space."""
    paths = []
    for p in glob.glob(os.path.join(logdir, "checkpoint", "state_*.pt")):
        m = re.search(r"state_(\d+)\.pt$", p)
        if m:
            paths.append((int(m.group(1)), p))
    paths.sort()
    for itr, p in paths[:-keep]:
        if itr != T["n_itr"] - 1:
            try:
                os.remove(p)
            except OSError:
                pass


def train_progress(job):
    log = tail(os.path.join(job["logdir"], "job.log"))
    train_itrs = [int(x) for x in re.findall(r"\] - (\d+): step\s+\d+", log)]
    evals = [float(x) for x in re.findall(r"eval: success rate\s+([0-9.]+)", log)]
    saved = last_ckpt_itr(job["logdir"])
    itr = max(train_itrs + ([saved] if saved is not None else []) + [-1])
    return itr, (evals[-1] if evals else None)


def eval_progress(job):
    log = tail(os.path.join(job["logdir"], "job.log"), 20_000)
    m = re.findall(r"Processed step (\d+) of (\d+)", log)
    return (int(m[-1][0]), int(m[-1][1])) if m else (0, None)


def eval_result(job):
    try:
        r = np.load(job["done_file"])
        n = int(r["num_episode"])
        k = int(round(float(r["eval_success_rate"]) * n))
        return f"{k}/{n} = {100.0 * k / max(n, 1):.1f}%"
    except Exception:
        return ""


# Status file
STATE = {j["name"]: dict(state="pending", tries=0, started=None, finished=None, seconds=None, note="") for j in JOBS}
if os.path.exists(STATUS_JSON):  # keep try counts and timings from earlier sessions
    try:
        old = json.load(open(STATUS_JSON)).get("jobs", {})
        for name, s in old.items():
            if name in STATE:
                # a job cut off by a disconnect (still "running") does not count as a failed try
                STATE[name]["tries"] = max(0, s.get("tries", 0) - (1 if s.get("state") == "running" else 0))
                STATE[name]["seconds"] = s.get("seconds")
                if s.get("state") == "failed":
                    STATE[name]["state"] = "failed"
                    STATE[name]["note"] = s.get("note", "")
    except Exception:
        pass
CURRENT = {}
HW = CFG.get("hardware", {})


def projected_finish():
    """Rough ETA for the remaining jobs, from timings measured in this study."""
    train_s = [s["seconds"] / T["n_itr"] for n, s in STATE.items() if n.startswith("train_") and s["state"] == "done" and s["seconds"]]
    eval_s = [s["seconds"] for n, s in STATE.items() if n.startswith("eval_") and s["state"] == "done" and s["seconds"]]
    per_itr = CURRENT.get("sec_per_itr") or (np.mean(train_s) if train_s else None)
    per_eval = np.mean(eval_s) if eval_s else None
    total = 0.0
    for j in JOBS:
        s = STATE[j["name"]]
        if s["state"] in ("done", "failed", "skipped", "blocked"):
            continue
        if j["name"] == CURRENT.get("name"):
            total += CURRENT.get("eta_s") or 0
        elif j["kind"] == "train":
            if per_itr is None:
                return None
            total += per_itr * T["n_itr"]
        else:
            if per_eval is None:
                return None
            total += per_eval
    return now() + dt.timedelta(seconds=total)


def write_status(final=False):
    rows = []
    for j in JOBS:
        s = STATE[j["name"]]
        if is_done(j) and s["state"] not in ("done",):
            s["state"] = "done"
        if skipped(j) and s["state"] == "pending":
            s["state"] = "skipped"
        info = s["note"]
        if s["state"] == "done" and j["kind"] in ("eval", "video"):
            info = eval_result(j)
        if j["name"] == CURRENT.get("name"):
            info = CURRENT.get("progress", "")
        dur = f"{s['seconds'] / 60:.1f} min" if s.get("seconds") else ""
        rows.append((j["name"], s["state"], s["tries"], dur, info))
    fin = projected_finish()
    counts = {k: sum(1 for r in rows if r[1] == k) for k in ("done", "running", "pending", "failed", "blocked", "skipped")}
    data = dict(role=ROLE, updated_at=stamp(), hardware=HW, current=CURRENT, counts=counts,
                projected_finish=stamp(fin) if fin else None, finished=final,
                jobs={j["name"]: STATE[j["name"]] for j in JOBS})
    tmp = STATUS_JSON + ".tmp"
    with open(tmp, "w") as f:
        json.dump(data, f, indent=1, default=str)
    os.replace(tmp, STATUS_JSON)
    lines = [
        f"# Pipeline status (role {ROLE})",
        f"Updated: {stamp()} (Phoenix time) | {'FINISHED' if final else 'running'}",
        f"Hardware: {HW.get('gpu', '?')} | {HW.get('cpus', '?')} vCPUs | {HW.get('ram_gb', '?')} GB RAM",
        f"Jobs: {counts['done']} done, {counts['running']} running, {counts['pending']} pending, "
        f"{counts['failed']} failed, {counts['blocked']} blocked, {counts['skipped']} skipped",
        f"Projected finish of everything: {stamp(fin) if fin else 'not known yet (needs one finished job of each type)'}",
        "",
    ]
    if CURRENT:
        lines += [f"Now running: {CURRENT['name']} | {CURRENT.get('progress', '')} | started {CURRENT.get('started', '')}",
                  "Last log lines:", "```", *CURRENT.get("tail", []), "```", ""]
    lines += ["| job | state | tries | time | result / progress |", "|---|---|---|---|---|"]
    lines += [f"| {a} | {b} | {c} | {d} | {e} |" for a, b, c, d, e in rows]
    tmp = STATUS_MD + ".tmp"
    with open(tmp, "w") as f:
        f.write("\n".join(lines) + "\n")
    os.replace(tmp, STATUS_MD)


# Running one job
def kill_group(p):
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        time.sleep(CFG.get("kill_grace_seconds", 10))
        os.killpg(os.getpgid(p.pid), signal.SIGKILL)
    except Exception:
        pass


def run_job(job):
    s = STATE[job["name"]]
    os.makedirs(job["logdir"], exist_ok=True)
    logpath = os.path.join(job["logdir"], "job.log")
    if job["kind"] == "train":
        # keep in-training eval history from earlier segments before a resume overwrites result.pkl
        rp = os.path.join(job["logdir"], "result.pkl")
        if os.path.exists(rp):
            n = len(glob.glob(os.path.join(job["logdir"], "result_seg*.pkl")))
            shutil.move(rp, os.path.join(job["logdir"], f"result_seg{n}.pkl"))
    s["state"], s["tries"], s["started"] = "running", s["tries"] + 1, stamp()
    start_itr = (last_ckpt_itr(job["logdir"]) or -1) + 1 if job["kind"] == "train" else 0
    CURRENT.clear()
    CURRENT.update(name=job["name"], started=stamp(), progress="starting")
    event(f"START {job['name']} (try {s['tries']})" + (f" from iteration {start_itr}" if start_itr else ""))
    t0 = time.time()
    with open(logpath, "a") as lf:
        lf.write(f"\n===== {stamp()} start try {s['tries']} =====\n{' '.join(job['cmd'])}\n")
        lf.flush()
        p = subprocess.Popen(job["cmd"], cwd=REPO, stdout=lf, stderr=subprocess.STDOUT, env=ENV, start_new_session=True)
        last_size, last_change = -1, time.time()
        while p.poll() is None:
            time.sleep(CFG.get("poll_seconds", 30))
            size = os.path.getsize(logpath)
            if size != last_size:
                last_size, last_change = size, time.time()
            elif time.time() - last_change > CFG["stall_minutes"] * 60:
                event(f"STALL {job['name']}: no log output for {CFG['stall_minutes']} min, killing it")
                kill_group(p)
                break
            if job["kind"] != "train" and time.time() - t0 > CFG["eval_timeout_minutes"] * 60:
                event(f"TIMEOUT {job['name']} after {CFG['eval_timeout_minutes']} min, killing it")
                kill_group(p)
                break
            el = time.time() - t0
            if job["kind"] == "train":
                itr, last_eval = train_progress(job)
                done_itrs = max(0, itr - start_itr + 1)
                spi = el / done_itrs if done_itrs > 0 else None
                eta = spi * (T["n_itr"] - itr - 1) if spi else None
                CURRENT.update(sec_per_itr=spi, eta_s=eta,
                               progress=f"iteration {max(itr, 0)}/{T['n_itr'] - 1}"
                               + (f", {spi / 60:.1f} min/iteration, ~{eta / 3600:.1f} h left" if spi else "")
                               + (f", last in-training eval success {100 * last_eval:.1f}%" if last_eval is not None else ""))
                prune_checkpoints(job["logdir"])
            else:
                step, total = eval_progress(job)
                CURRENT.update(eta_s=None, progress=f"policy step {step}/{total or '?'}")
            CURRENT["tail"] = [ln[:160] for ln in tail(logpath, 4000).strip().splitlines()[-6:]]
            write_status()
    rc = p.wait()
    secs = time.time() - t0
    ok = rc == 0 and (job["kind"] != "train" or os.path.exists(job["final_ckpt"]))
    if ok and job["kind"] == "train":
        prev = (s.get("seconds") or 0) if start_itr > 0 else 0
        s["seconds"] = prev + secs
        with open(job["done_file"], "w") as f:
            json.dump(dict(finished=stamp(), seconds_this_segment=secs, total_seconds=s["seconds"], hardware=HW, cmd=job["cmd"]), f, indent=1)
        prune_checkpoints(job["logdir"])
    elif ok:
        s["seconds"] = secs
    ok = ok and is_done(job)
    if ok:
        s["state"], s["finished"], s["note"] = "done", stamp(), ""
        event(f"DONE  {job['name']} in {secs / 60:.1f} min " + (eval_result(job) if job["kind"] != "train" else ""))
    else:
        err = [ln for ln in tail(logpath, 30000).splitlines() if re.search(r"Error|Exception|Traceback|Killed|CUDA", ln)]
        s["note"] = (err[-1][:200] if err else f"exit code {rc}")
        s["state"] = "failed" if s["tries"] >= CFG["max_tries"] else "pending"
        if job["kind"] == "train":  # keep the time spent so a resume can add to it
            s["seconds"] = (s.get("seconds") or 0) + secs
        event(f"FAIL  {job['name']} (try {s['tries']}/{CFG['max_tries']}, exit {rc}): {s['note']}")
    CURRENT.clear()
    write_status()
    return ok


# Main loop
def main():
    event(f"pipeline started, role {ROLE}, {len(JOBS)} jobs")
    for j in JOBS:  # a job that failed in an earlier session gets a fresh chance after a restart
        if STATE[j["name"]]["state"] == "failed" and STATE[j["name"]]["tries"] < CFG["max_tries"] + 1:
            STATE[j["name"]]["state"] = "pending"
    while True:
        if os.path.exists(os.path.join(PIPE_DIR, "STOP")):
            event("STOP file found, exiting")
            break
        write_status()
        todo = [j for j in JOBS if not is_done(j) and not skipped(j) and STATE[j["name"]]["state"] == "pending"]
        if not todo:
            break
        runnable, waiting = [], False
        for j in todo:
            deps_ok = all(is_done(d) for d in j["deps"])
            if deps_ok:
                runnable.append(j)
                continue
            dead = [d for d in j["deps"] if d in ALL_NAMES and STATE[d]["state"] in ("failed", "skipped")]
            if dead:
                STATE[j["name"]]["state"], STATE[j["name"]]["note"] = "blocked", f"needs {dead[0]}"
            else:
                waiting = True  # dependency still running here or on the other runtime
        if runnable:
            run_job(runnable[0])
        elif waiting:
            time.sleep(CFG.get("wait_seconds", 300))
        else:
            break
    write_status(final=True)
    event("pipeline finished")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/work/analyze.py
"""
Collects every finished evaluation and training run into tables, figures and a short
text summary. Safe to run at any time, it only reports what has actually finished.

    python analyze.py config.json
Outputs go to <project>/analysis/.
"""

import csv
import glob
import json
import math
import os
import pickle
import sys
from collections import defaultdict

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

CFG = json.load(open(sys.argv[1]))
P = CFG["project"]
OUT = os.path.join(P, "analysis")
os.makedirs(OUT, exist_ok=True)
CONDS = {c["name"]: c for c in CFG["conditions"]}
COND_ORDER = [c["name"] for c in CFG["conditions"]]

METHOD_LABEL = {
    "pretrained": "Pretrained diffusion policy",
    "pretrained_raw": "Pretrained (raw weights)",
    "dppo": "DPPO",
    "robust": "Robust DPPO",
}
COLORS = {"pretrained": "#7f7f7f", "dppo": "#1f77b4", "robust": "#d62728", "pretrained_raw": "#bcbd22"}


def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))


def split_model(model):
    if model in ("pretrained", "pretrained_raw"):
        return model, None
    kind, seed = model.rsplit("_s", 1)
    return kind, int(seed)


# 1) Evaluation results
rows = []
for path in sorted(glob.glob(os.path.join(P, "runs", "eval", "*", "*", "result.npz"))):
    cond_name = os.path.basename(os.path.dirname(path))
    model = os.path.basename(os.path.dirname(os.path.dirname(path)))
    if cond_name not in CONDS:
        continue
    r = np.load(path)
    n = int(r["num_episode"])
    k = int(round(float(r["eval_success_rate"]) * n))
    lo, hi = wilson(k, n)
    method, seed = split_model(model)
    c = CONDS[cond_name]
    rows.append(dict(model=model, method=method, seed=seed if seed is not None else "", condition=cond_name,
                     type=c["type"], level=c["level"], episodes=n, successes=k, success_rate=k / max(n, 1),
                     ci95_low=lo, ci95_high=hi, source=os.path.relpath(path, P)))

with open(os.path.join(OUT, "results_all_evaluations.csv"), "w", newline="") as f:
    cols = ["model", "method", "seed", "condition", "type", "level", "episodes", "successes", "success_rate",
            "ci95_low", "ci95_high", "source"]
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    for r in rows:
        w.writerow({k: (f"{v:.4f}" if isinstance(v, float) else v) for k, v in r.items()})

# pool seeds per method and condition
pooled = defaultdict(lambda: dict(k=0, n=0, seeds=[], rates=[]))
for r in rows:
    if r["method"] == "pretrained_raw":
        continue
    key = (r["method"], r["condition"])
    pooled[key]["k"] += r["successes"]
    pooled[key]["n"] += r["episodes"]
    pooled[key]["seeds"].append(r["seed"])
    pooled[key]["rates"].append(r["success_rate"])
methods = [m for m in ("pretrained", "dppo", "robust") if any(k[0] == m for k in pooled)]


def rate(method, cond):
    d = pooled.get((method, cond))
    if not d or d["n"] == 0:
        return None
    return d["k"] / d["n"]


def cell(method, cond):
    d = pooled.get((method, cond))
    if not d or d["n"] == 0:
        return "not run"
    lo, hi = wilson(d["k"], d["n"])
    s = f"{100 * d['k'] / d['n']:.0f}% ({d['k']}/{d['n']}) [{100 * lo:.0f}-{100 * hi:.0f}]"
    if len(d["rates"]) > 1:
        s += f", seeds {np.mean(d['rates']) * 100:.0f}+/-{np.std(d['rates'], ddof=1) * 100:.0f}"
    return s


def diff_ci(m1, m2, cond):
    """Difference m1 - m2 in percentage points with a normal-approximation 95% interval."""
    a, b = pooled.get((m1, cond)), pooled.get((m2, cond))
    if not a or not b or a["n"] == 0 or b["n"] == 0:
        return None
    p1, p2 = a["k"] / a["n"], b["k"] / b["n"]
    se = math.sqrt(p1 * (1 - p1) / a["n"] + p2 * (1 - p2) / b["n"])
    return 100 * (p1 - p2), 100 * 1.96 * se


# main table (markdown)
md = ["| Condition | " + " | ".join(METHOD_LABEL[m] for m in methods) + " |",
      "|---|" + "---|" * len(methods)]
for cn in COND_ORDER:
    md.append(f"| {CONDS[cn]['label']} | " + " | ".join(cell(m, cn) for m in methods) + " |")
main_table = "\n".join(md)
with open(os.path.join(OUT, "table_main.md"), "w") as f:
    f.write("Success rate (successes/episodes) [95% Wilson interval]\n\n" + main_table + "\n")

# 2) Figures
figs = []
if rows:
    # nominal bar chart
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    xs, labels = [], []
    for i, m in enumerate(methods):
        d = pooled.get((m, "nominal"))
        if not d:
            continue
        p = d["k"] / d["n"]
        lo, hi = wilson(d["k"], d["n"])
        ax.bar(i, 100 * p, color=COLORS[m], width=0.6)
        ax.errorbar(i, 100 * p, yerr=[[100 * (p - lo)], [100 * (hi - p)]], color="k", capsize=4)
        ax.text(i, 100 * hi + 2, f"{100 * p:.0f}%", ha="center", fontsize=9)
        xs.append(i)
        labels.append(METHOD_LABEL[m])
    ax.set_xticks(xs)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("Success rate (%)")
    ax.set_ylim(0, 105)
    ax.set_title("Robomimic Square, nominal conditions (95% CI)", fontsize=10)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, "fig_nominal_success.png"), dpi=200)
    fig.savefig(os.path.join(OUT, "fig_nominal_success.pdf"))
    plt.close(fig)
    figs.append("fig_nominal_success.png")

    # robustness curves, one panel per perturbation type
    types = []
    for cn in COND_ORDER:
        t = CONDS[cn]["type"]
        if t != "Nominal" and t not in types:
            types.append(t)
    fig, axes = plt.subplots(1, len(types), figsize=(3.3 * len(types), 3.3), squeeze=False)
    for ax, t in zip(axes[0], types):
        levels = [(CONDS["nominal"]["level_for"].get(t), "nominal")] + [
            (CONDS[cn]["level"], cn) for cn in COND_ORDER if CONDS[cn]["type"] == t]
        levels.sort(key=lambda x: x[0])
        for m in methods:
            pts = [(lv, rate(m, cn), pooled.get((m, cn))) for lv, cn in levels if rate(m, cn) is not None]
            if not pts:
                continue
            x = [p[0] for p in pts]
            y = [100 * p[1] for p in pts]
            err = np.array([[100 * (p[1] - wilson(p[2]["k"], p[2]["n"])[0]), 100 * (wilson(p[2]["k"], p[2]["n"])[1] - p[1])] for p in pts]).T
            ax.errorbar(x, y, yerr=err, marker="o", capsize=3, color=COLORS[m], label=METHOD_LABEL[m])
        ax.set_title(t, fontsize=10)
        ax.set_xlabel(CONDS[[cn for cn in COND_ORDER if CONDS[cn]["type"] == t][0]]["unit"])
        ax.set_ylim(0, 105)
        ax.grid(alpha=0.3)
    axes[0][0].set_ylabel("Success rate (%)")
    axes[0][-1].legend(fontsize=8, loc="lower left")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, "fig_robustness.png"), dpi=200)
    fig.savefig(os.path.join(OUT, "fig_robustness.pdf"))
    plt.close(fig)
    figs.append("fig_robustness.png")

# learning curves from in-training evaluations
train_runs = {}
for d in sorted(glob.glob(os.path.join(P, "runs", "train", "*"))):
    name = os.path.basename(d)
    hist = {}
    for pk in sorted(glob.glob(os.path.join(d, "result_seg*.pkl"))) + glob.glob(os.path.join(d, "result.pkl")):
        try:
            for r in pickle.load(open(pk, "rb")):
                hist.setdefault(r["itr"], {}).update(r)
        except Exception:
            pass
    done = os.path.exists(os.path.join(d, "DONE.json"))
    secs = json.load(open(os.path.join(d, "DONE.json"))).get("total_seconds") if done else None
    evals = sorted((r["itr"], r["step"], r["eval_success_rate"]) for r in hist.values() if "eval_success_rate" in r)
    last_step = max((r["step"] for r in hist.values()), default=0)
    train_runs[name] = dict(done=done, seconds=secs, evals=evals, last_itr=max(hist) if hist else None, env_steps=last_step)

if any(v["evals"] for v in train_runs.values()):
    fig, ax = plt.subplots(figsize=(5.6, 3.5))
    for name, v in train_runs.items():
        if not v["evals"]:
            continue
        kind = name.rsplit("_s", 1)[0]
        x = [s / 1e6 for _, s, _ in v["evals"]]
        y = [100 * e for _, _, e in v["evals"]]
        style = "-" if name.endswith(f"_s{CFG['seeds_main'][0]}") else "--"
        lab = f"{METHOD_LABEL[kind]} (seed {name.rsplit('_s', 1)[1]})" + (" - evaluated with randomization on" if kind == "robust" else "")
        ax.plot(x, y, style, marker="o", color=COLORS[kind], label=lab)
    ax.set_xlabel("Environment steps used for fine-tuning (millions)")
    ax.set_ylabel("In-training eval success (%)")
    ax.set_ylim(0, 105)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
    ax.set_title("Fine-tuning curves", fontsize=10)
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, "fig_learning_curves.png"), dpi=200)
    fig.savefig(os.path.join(OUT, "fig_learning_curves.pdf"))
    plt.close(fig)
    figs.append("fig_learning_curves.png")

# 3) Plain-language summary with the numbers filled in
perturbed = [cn for cn in COND_ORDER if cn != "nominal"]


def avg_perturbed(m):
    vals = [rate(m, cn) for cn in perturbed]
    vals = [v for v in vals if v is not None]
    return (100 * np.mean(vals), len(vals)) if vals else (None, 0)


def worst(m):
    vals = [(rate(m, cn), cn) for cn in perturbed if rate(m, cn) is not None]
    return min(vals) if vals else None


lines = ["# Results summary (auto-generated from finished runs only)", ""]
hw = CFG.get("hardware", {})
lines += [f"Hardware: {hw.get('gpu', '?')}, {hw.get('cpus', '?')} vCPUs, {hw.get('ram_gb', '?')} GB RAM (Google Colab).",
          f"Evaluation: {CFG['eval']['n_envs'] * CFG['eval']['episodes_per_env']} episodes per condition, eval seed {CFG['eval']['seed']}, "
          "same initial states for every method; 95% Wilson intervals.", ""]
lines += ["## Main table", "", main_table, ""]

raw = [r for r in rows if r["model"] == "pretrained_raw" and r["condition"] == "nominal"]
ema = pooled.get(("pretrained", "nominal"))
if raw or ema:
    lines += ["## Sanity check: raw vs EMA weights", ""]
    if raw:
        r = raw[0]
        lines.append(f"- Pretrained checkpoint, raw weights (what DPPO's eval script loads): {100 * r['success_rate']:.0f}% "
                     f"({r['successes']}/{r['episodes']}).")
    if ema:
        lines.append(f"- Same checkpoint, EMA weights (the policy DPPO actually starts from): {100 * ema['k'] / ema['n']:.0f}% "
                     f"({ema['k']}/{ema['n']}). This is the baseline used in the main table.")
    lines.append("")

lines += ["## Training runs", "", "| run | finished | iterations logged | env steps | wall time | in-training eval success (first -> last) |", "|---|---|---|---|---|---|"]
for name, v in train_runs.items():
    ev = v["evals"]
    curve = f"{100 * ev[0][2]:.0f}% -> {100 * ev[-1][2]:.0f}%" if ev else "-"
    wt = f"{v['seconds'] / 3600:.1f} h" if v["seconds"] else "-"
    lines.append(f"| {name} | {'yes' if v['done'] else 'no'} | {v['last_itr']} | {v['env_steps']:,} | {wt} | {curve} |")
lines.append("")

facts = []
dn = diff_ci("dppo", "pretrained", "nominal")
if dn:
    facts.append(f"DPPO vs pretrained, nominal: {dn[0]:+.0f} percentage points (+/- {dn[1]:.0f}, 95%).")
for m in methods:
    a, k = avg_perturbed(m)
    wv = worst(m)
    if a is not None:
        facts.append(f"{METHOD_LABEL[m]}: mean success over {k} perturbed conditions {a:.0f}%"
                     + (f"; worst case {100 * wv[0]:.0f}% ({CONDS[wv[1]]['label']})." if wv else "."))
for cn in perturbed:
    d = diff_ci("dppo", "pretrained", cn)
    if d:
        facts.append(f"DPPO - pretrained under {CONDS[cn]['label']}: {d[0]:+.0f} pp (+/- {d[1]:.0f}).")
for cn in COND_ORDER:
    d = diff_ci("robust", "dppo", cn)
    if d:
        facts.append(f"Robust DPPO - DPPO under {CONDS[cn]['label']}: {d[0]:+.0f} pp (+/- {d[1]:.0f}).")
lines += ["## Key numbers (check each one against the table before using it)", ""] + [f"- {x}" for x in facts] + [""]
lines += ["## Figures", ""] + [f"- {f}" for f in figs] + [""]
lines += ["## Files", "", "- results_all_evaluations.csv: one row per evaluation run, with its source file", "- table_main.md: the main table", ""]

with open(os.path.join(OUT, "RESULTS_SUMMARY.md"), "w") as f:
    f.write("\n".join(lines))
with open(os.path.join(OUT, "key_numbers.json"), "w") as f:
    json.dump(dict(pooled={f"{m}|{c}": v for (m, c), v in pooled.items()}, train_runs=train_runs, facts=facts), f, indent=1, default=str)
print("\n".join(lines))

In [ ]:
CONFIG = dict(role=ROLE, project=PROJECT, repo=REPO, python=VPY, env_vars=ENV_VARS, timezone="America/Phoenix",
              train=TRAIN, seeds_main=SEEDS_MAIN, seeds_extra=SEEDS_EXTRA, eval=EVAL, dr=DR, conditions=CONDITIONS,
              pretrained_ckpt=PRE, pretrained_ema_ckpt=PRE_EMA, ft_denoising_steps=10, max_episode_steps=400,
              act_steps=4, stall_minutes=25, eval_timeout_minutes=60, max_tries=2, hardware=HARDWARE)
CFG_PATH = f"{PROJECT}/pipeline/config_{ROLE}.json"
json.dump(CONFIG, open(CFG_PATH, "w"), indent=1)
for f in ["pipeline.py", "analyze.py"]:
    shutil.copy(f"/content/work/{f}", f"{PROJECT}/code/{f}")
PIDFILE = f"/content/work/pipeline_{ROLE}.pid"


def pipeline_alive():
    try:
        os.kill(int(open(PIDFILE).read()), 0)
        return True
    except Exception:
        return False


def launch():
    log = open(f"{PROJECT}/pipeline/pipeline_{ROLE}.out", "a")
    p = subprocess.Popen([VPY, "/content/work/pipeline.py", CFG_PATH], stdout=log, stderr=subprocess.STDOUT,
                         start_new_session=True)
    open(PIDFILE, "w").write(str(p.pid))
    print("pipeline started, pid", p.pid)


if pipeline_alive():
    print("pipeline already running")
else:
    launch()

### 8. Live monitor (keeps the notebook busy, restarts the pipeline if it dies, refreshes every 5 min)

In [ ]:
from IPython.display import clear_output, Markdown, display

STATUS = f"{PROJECT}/pipeline/STATUS_{ROLE}.md"
restarts, t_end = 0, time.time() + 23.5 * 3600
while time.time() < t_end:
    clear_output(wait=True)
    try:
        display(Markdown(open(STATUS).read()))
    except FileNotFoundError:
        print("waiting for the first status file...")
    try:
        done = json.load(open(f"{PROJECT}/pipeline/status_{ROLE}.json")).get("finished")
    except Exception:
        done = False
    if os.path.exists(f"{PROJECT}/pipeline/STOP"):
        print("STOP file present.")
    if done:
        print("PIPELINE FINISHED - run Part 2.")
        break
    if not pipeline_alive() and not os.path.exists(f"{PROJECT}/pipeline/STOP"):
        restarts += 1
        print(f"pipeline process not running - restarting (#{restarts})")
        launch()
    print(f"monitor heartbeat {time.strftime('%H:%M:%S')} UTC | pipeline restarts this session: {restarts}")
    time.sleep(300)

## Part 2 - Results: tables, figures and videos
Run these any time, even on a fresh runtime. The first cell only needs Drive.

In [ ]:
import os, json, subprocess, sys, glob
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DPPO_Robustness"
ROLE_CFG = sorted(glob.glob(f"{PROJECT}/pipeline/config_*.json"))[0]
os.makedirs("/content/work", exist_ok=True)
cfg = json.load(open(ROLE_CFG))
cfg["project"] = PROJECT
json.dump(cfg, open("/content/work/analysis_config.json", "w"))
r = subprocess.run([sys.executable, f"{PROJECT}/code/analyze.py", "/content/work/analysis_config.json"],
                   capture_output=True, text=True)
print(r.stderr[-2000:])
from IPython.display import Markdown, display, Image
display(Markdown(open(f"{PROJECT}/analysis/RESULTS_SUMMARY.md").read()))
for f in ["fig_nominal_success.png", "fig_robustness.png", "fig_learning_curves.png"]:
    p = f"{PROJECT}/analysis/{f}"
    if os.path.exists(p):
        display(Image(p, width=900))

### Demo videos (four episodes per clip set; the result line says how many succeeded)

In [ ]:
import base64
from IPython.display import HTML
for d in sorted(glob.glob(f"{PROJECT}/runs/video/*/*")):
    vids = sorted(glob.glob(f"{d}/render/*.mp4"))
    if not vids:
        continue
    import numpy as np
    r = np.load(f"{d}/result.npz") if os.path.exists(f"{d}/result.npz") else None
    head = f"{d.split('/runs/video/')[1]}" + (f" - success {float(r['eval_success_rate']) * 100:.0f}% of {int(r['num_episode'])}" if r is not None else "")
    html = f"<h4>{head}</h4>" + "".join(
        f'<video width="240" controls src="data:video/mp4;base64,{base64.b64encode(open(v, "rb").read()).decode()}"></video>'
        for v in vids)
    display(HTML(html))

### Package everything into one zip on Drive

In [ ]:
import zipfile
out = f"{PROJECT}/dppo_robustness_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in ["analysis/*", "pipeline/*.md", "pipeline/*.json", "pipeline/events_*.log", "code/*",
                    "runs/eval/*/*/result.npz", "runs/train/*/DONE.json", "runs/train/*/result*.pkl",
                    "runs/train/*/job.log", "runs/video/*/*/render/*.mp4"]:
        for f in glob.glob(f"{PROJECT}/{pattern}"):
            z.write(f, os.path.relpath(f, PROJECT))
print(out, round(os.path.getsize(out) / 1e6, 1), "MB")